# 03 - Feature selection results

Exploratory companion to `scripts/select_features.py`. Run
`python scripts/run_pipeline.py --stage select_features` first, then use this notebook to
compare the permutation-importance sweep and RFECV's chosen feature counts per fold, and how
much they agree.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from cfb_spread_model.utils.paths import OUTPUTS_FEATURE_ANALYSIS

with open(OUTPUTS_FEATURE_ANALYSIS / "feature_selection_summary.json") as f:
    summary = json.load(f)

pd.DataFrame(summary)[
    ["validation_season", "n_columns_before_pruning", "n_columns_after_pruning", "permutation_sweep_best_n", "rfecv_n_features"]
]

In [ ]:
# Overlap between the permutation-importance sweep and RFECV's selected features, per fold
for row in summary:
    perm = set(row["selected_features"])
    rfecv = set(row.get("rfecv_features") or [])
    if not rfecv:
        continue
    overlap = len(perm & rfecv) / max(1, len(perm | rfecv))
    print(f"fold {row['validation_season']}: perm={len(perm)}, rfecv={len(rfecv)}, jaccard overlap={overlap:.2f}")

In [ ]:
# Permutation-importance sweep table for a single fold
fold_season = summary[0]["validation_season"]
pd.read_csv(OUTPUTS_FEATURE_ANALYSIS / f"permutation_sweep_fold_{fold_season}.csv")